# Подключение библиотек

In [1]:
import pandas as pd
import datetime
from datetime import date, timedelta
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Загрузка данных

In [2]:
control_data = pd.read_csv("/content/control_group.csv", sep = ";")
test_data = pd.read_csv("/content/test_group.csv", sep = ";")
print(control_data.head())
print('-' * 100)
print(test_data.head())

      Campaign Name       Date  Spend [USD]  # of Impressions     Reach  \
0  Control Campaign  1.08.2019         2280           82702.0   56930.0   
1  Control Campaign  2.08.2019         1757          121040.0  102513.0   
2  Control Campaign  3.08.2019         2343          131711.0  110862.0   
3  Control Campaign  4.08.2019         1940           72878.0   61235.0   
4  Control Campaign  5.08.2019         1835               NaN       NaN   

   # of Website Clicks  # of Searches  # of View Content  # of Add to Cart  \
0               7016.0         2290.0             2159.0            1819.0   
1               8110.0         2033.0             1841.0            1219.0   
2               6508.0         1737.0             1549.0            1134.0   
3               3065.0         1042.0              982.0            1183.0   
4                  NaN            NaN                NaN               NaN   

   # of Purchase  
0          618.0  
1          511.0  
2          372.0  
3   

# Предобработка данных

В исходных данных были обнаружены ошибки в названиях колонок. Перед началом анализа названия были приведены к единому формату.

In [3]:
control_data.columns = ["Campaign Name", "Date", "Amount Spent",
                        "Number of Impressions", "Reach", "Website Clicks",
                        "Searches Received", "Content Viewed", "Added to Cart",
                        "Purchases"]

test_data.columns = ["Campaign Name", "Date", "Amount Spent",
                        "Number of Impressions", "Reach", "Website Clicks",
                        "Searches Received", "Content Viewed", "Added to Cart",
                        "Purchases"]

# Проверка данных

Проверим наличие пропущенных значений в данных.

In [4]:
null_rows = control_data[control_data.isnull().any(axis=1)]
null_row_indices = control_data[control_data.isnull().any(axis=1)].index

print("Rows with missing values:")
print(null_rows)
print("\nRow indices with missing values:", null_row_indices.tolist())

Rows with missing values:
      Campaign Name       Date  Amount Spent  Number of Impressions  Reach  \
4  Control Campaign  5.08.2019          1835                    NaN    NaN   

   Website Clicks  Searches Received  Content Viewed  Added to Cart  Purchases  
4             NaN                NaN             NaN            NaN        NaN  

Row indices with missing values: [4]


В данных обнаружена одна строка с пропущенными значениями, где заполнено только поле с затратами.

Такая ситуация может быть связана с ошибкой при загрузке данных или некорректным разделителем.

Поскольку каждая запись соответствует отдельному дню кампании и может иметь уникальные характеристики, заполнение пропусков средними значениями может исказить результаты.

Поэтому данная строка была удалена.

In [5]:
control_data = control_data.dropna()
print("\nShape of control_data after dropping missing values:", control_data.shape)


Shape of control_data after dropping missing values: (29, 10)


In [6]:
print(test_data.isnull().sum())

Campaign Name            0
Date                     0
Amount Spent             0
Number of Impressions    0
Reach                    0
Website Clicks           0
Searches Received        0
Content Viewed           0
Added to Cart            0
Purchases                0
dtype: int64


Для удобства дальнейшего анализа создадим отдельные датафреймы для контрольной и тестовой групп.

In [7]:
control = pd.DataFrame(control_data)
test = pd.DataFrame(test_data)

### Выбор метрик

Для анализа выбраны метрики, отражающие ключевые этапы пользовательской воронки:

- Website Clicks — первичная вовлечённость
- Searches Received — интерес к продукту
- Added to Cart — намерение покупки
- Purchases — итоговая конверсия

Дополнительно рассчитаны производные метрики:

- Conversion Rate — эффективность конверсии
- Cost per Conversion — стоимость конверсии
- Content Viewed — вовлечённость в контент
- CPA (Cost per Acquisition) = затраты / покупки — стоимость привлечения
- CTR (Click-Through Rate) = клики / показы — эффективность рекламных креативов

Совокупность этих метрик позволяет комплексно оценить эффективность кампаний на всех этапах воронки.

In [8]:
metrics = ['Website Clicks', 'Searches Received', 'Added to Cart', 'Purchases', 'Conversion Rate (Purchase)', 'Cost per Conversion', 'Content Viewed', 'CPA', 'CTR']

### Почему выбран t-test

- T-test:
  используется для сравнения средних значений двух групп (Control и Test), особенно при небольших выборках и неизвестных дисперсиях. В данном случае применяется t-test Уэлча, который не требует равенства дисперсий.

- Z-test:
  требует больших выборок и известных параметров генеральной совокупности, что не выполняется в данном случае.

- Другие тесты:
  непараметрические методы (например, Mann–Whitney) могут использоваться при ненормальном распределении, однако для агрегированных метрик допущение нормальности является приемлемым.

Таким образом, t-test является наиболее подходящим методом для данного анализа.

# Проведение A/B-теста

Функция для расчёта производных метрик (CPA, CTR и др.) и обработки некорректных значений.

Функция для предобработки данных и расчёта метрик по каждой строке (CPA, CTR и др.).

In [9]:
def preprocess_data(data):
    data = data.copy()
    data['CPA'] = data['Amount Spent'] / data['Purchases']
    data['CPA'] = data['CPA'].replace([np.inf, -np.inf], np.nan).fillna(data['CPA'].mean())
    data['CTR'] = data['Website Clicks'] / data['Number of Impressions']
    data['CTR'] = data['CTR'].replace([np.inf, -np.inf], np.nan).fillna(data['CTR'].mean())
    data['Conversion Rate (Purchase)'] = data['Purchases'] / data['Website Clicks']
    data['Conversion Rate (Purchase)'] = data['Conversion Rate (Purchase)'].replace([np.inf, -np.inf], np.nan).fillna(data['Conversion Rate (Purchase)'].mean())
    data['Cost per Conversion'] = data['Amount Spent'] / data['Purchases']
    data['Cost per Conversion'] = data['Cost per Conversion'].replace([np.inf, -np.inf], np.nan).fillna(data['Cost per Conversion'].mean())
    return data

Функция для проведения t-test и расчёта статистических показателей (p-value, effect size).

In [10]:
def perform_ab_test(control_data, test_data, metric):
    t_stat, p_value = stats.ttest_ind(control_data[metric], test_data[metric], equal_var=False, nan_policy='omit')
    control_mean = control_data[metric].mean()
    test_mean = test_data[metric].mean()
    effect_size = (test_mean - control_mean) / control_data[metric].std() if control_data[metric].std() != 0 else 0
    return {
        'metric': metric,
        'control_mean': control_mean,
        'test_mean': test_mean,
        't_stat': t_stat,
        'p_value': p_value,
        'effect_size': effect_size
    }

In [11]:
control = preprocess_data(control)
test = preprocess_data(test)

In [12]:
results = []
for metric in metrics:
    if metric in control.columns:
        result = perform_ab_test(control, test, metric)
        if result:
            results.append(result)

print("A/B Testing Results:")
print("-" * 50)
for result in results:
    print(f"\nMetric: {result['metric']}")
    print(f"Control Mean: {result['control_mean']:.2f}")
    print(f"Test Mean: {result['test_mean']:.2f}")
    print(f"P-value: {result['p_value']:.4f}")
    print(f"Effect Size: {result['effect_size']:.2f}")
    print("Statistically Significant" if result['p_value'] < 0.05 else "Not Statistically Significant")

print("\nConclusions:")
print("-" * 50)
print("Based on the analysis:")
for result in results:
    if result['p_value'] < 0.05:
        print(f"- {result['metric']}: Significant difference found. {'Test' if result['test_mean'] > result['control_mean'] else 'Control'} performs better.")
    else:
        print(f"- {result['metric']}: No significant difference found.")

A/B Testing Results:
--------------------------------------------------

Metric: Website Clicks
Control Mean: 5320.79
Test Mean: 6032.33
P-value: 0.1205
Effect Size: 0.40
Not Statistically Significant

Metric: Searches Received
Control Mean: 2221.31
Test Mean: 2418.97
P-value: 0.2678
Effect Size: 0.23
Not Statistically Significant

Metric: Added to Cart
Control Mean: 1300.00
Test Mean: 881.53
P-value: 0.0001
Effect Size: -1.03
Statistically Significant

Metric: Purchases
Control Mean: 522.79
Test Mean: 521.23
P-value: 0.9760
Effect Size: -0.01
Not Statistically Significant

Metric: Conversion Rate (Purchase)
Control Mean: 0.11
Test Mean: 0.09
P-value: 0.1428
Effect Size: -0.33
Not Statistically Significant

Metric: Cost per Conversion
Control Mean: 5.05
Test Mean: 5.90
P-value: 0.1946
Effect Size: 0.40
Not Statistically Significant

Metric: Content Viewed
Control Mean: 1943.79
Test Mean: 1858.00
P-value: 0.6374
Effect Size: -0.11
Not Statistically Significant

Metric: CPA
Control Mean:

### Рекомендуемая кампания:
Test Campaign


### Обоснование

Тестовая кампания показывает значительное преимущество по CTR, что говорит о более высокой эффективности в привлечении пользователей и генерации трафика.

Несмотря на то, что контрольная кампания показывает лучшие результаты по добавлению в корзину, различия в ключевых метриках — покупках и стоимости привлечения (CPA) — статистически незначимы.

Это означает, что увеличение трафика в тестовой кампании не приводит к ухудшению конверсии или росту затрат.

Поэтому тестовая кампания рекомендуется к использованию за счёт более высокой эффективности на верхнем уровне воронки без негативного влияния на бизнес-метрики.
